# 03 — Gold

One row per ZIP. **This is the index.** Silver stays unjoined facts. Gold asks: when someone calls, what can we honestly say about service?

**We cannot prove the city showed up.** There is no visit log, no resolution text, and no link from a 311 ticket to a kitchen inspection.

| Table | Grain | Job |
|---|---|---|
| `gold_borough_index` | one borough | Pooled SGI (sum of tickets, not average of ZIP scores) |
| `gold_zip_index` | one ZIP | SGI-30 + ZIP card (gap split, character, cluster flags) |
| `gold_zip_leaderboard` | rank-eligible ZIPs only | View for top-N. Never titled worst rats. |
| `gold_zip_month` | ZIP × month | Season + right-censoring. Do not average monthly SGI. |
| `gold_citywide_week` | one week | Light viral-week sparkline. No ZIP-week model. |
| `gold_run_meta` | one row | Cutoff, hidden trap, pooled vs unweighted SGI |
| `gold_lookup_contract` | one row | Genie + dashboard instructions |

**SGI-30** = share of *mature* tickets with no recorded closure within 30 days. Higher = bigger paperwork gap. Not a rat census. Not a visit rate. Tickets younger than 30 days are left out of the score (right-censoring). Instant administrative stamps (`closed_ts = created_ts`) still count as K; they are disclosed, not dropped.

**Rank set:** `zip_type = neighborhood` AND mature N ≥ 30. Airports, `101xx` except 10128, NJ, and thin ZIPs get a lookup card, not a leaderboard slot. **`zip_character` is a card label, not the rank gate.**

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_borough_index AS
WITH params AS (
  SELECT MAX(created_ts) AS observation_cutoff
  FROM workspace.default.silver_rat_sightings
),
t AS (
  SELECT
    r.borough,
    r.created_ts,
    r.closed_ts,
    r.is_same_timestamp_close,
    p.observation_cutoff,
    CASE WHEN r.dq_closed_flag IN ('parse_fail', 'negative_duration') THEN TRUE ELSE FALSE END AS is_outcome_excl
  FROM workspace.default.silver_rat_sightings r
  CROSS JOIN params p
  WHERE r.created_ts IS NOT NULL
    AND r.complaint_type = 'Rodent'
    AND r.borough IN ('BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND')
),
t2 AS (
  SELECT
    borough,
    CASE
      WHEN created_ts <= observation_cutoff - INTERVAL 30 DAYS AND NOT is_outcome_excl THEN 1
      ELSE 0
    END AS in_N,
    CASE
      WHEN created_ts <= observation_cutoff - INTERVAL 30 DAYS
           AND NOT is_outcome_excl
           AND closed_ts IS NOT NULL
           AND closed_ts >= created_ts
           AND closed_ts <= created_ts + INTERVAL 30 DAYS
           AND closed_ts <= observation_cutoff THEN 1
      ELSE 0
    END AS in_K,
    CASE
      WHEN created_ts <= observation_cutoff - INTERVAL 30 DAYS
           AND NOT is_outcome_excl
           AND NOT (
             closed_ts IS NOT NULL
             AND closed_ts >= created_ts
             AND closed_ts <= created_ts + INTERVAL 30 DAYS
             AND closed_ts <= observation_cutoff
           )
           AND closed_ts IS NULL THEN 1
      ELSE 0
    END AS in_gap_still_open,
    CASE
      WHEN created_ts <= observation_cutoff - INTERVAL 30 DAYS
           AND NOT is_outcome_excl
           AND NOT (
             closed_ts IS NOT NULL
             AND closed_ts >= created_ts
             AND closed_ts <= created_ts + INTERVAL 30 DAYS
             AND closed_ts <= observation_cutoff
           )
           AND closed_ts IS NOT NULL THEN 1
      ELSE 0
    END AS in_gap_late_closed,
    CASE
      WHEN created_ts <= observation_cutoff - INTERVAL 30 DAYS
           AND NOT is_outcome_excl
           AND closed_ts IS NOT NULL
           AND closed_ts >= created_ts
           AND closed_ts <= created_ts + INTERVAL 30 DAYS
           AND closed_ts <= observation_cutoff
           AND is_same_timestamp_close THEN 1
      ELSE 0
    END AS in_k_instant,
    1 AS n_311
  FROM t
),
agg AS (
  SELECT
    borough,
    SUM(n_311) AS n_311,
    SUM(in_N) AS n_mature,
    SUM(in_K) AS n_timely_close_30d,
    SUM(in_N) - SUM(in_K) AS n_gap_30d,
    SUM(in_gap_still_open) AS n_gap_still_open,
    SUM(in_gap_late_closed) AS n_gap_late_closed,
    SUM(in_k_instant) AS n_k_instant,
    CASE
      WHEN SUM(in_N) >= 30 THEN 100.0 * (SUM(in_N) - SUM(in_K)) / SUM(in_N)
      ELSE NULL
    END AS sgi_30,
    CASE
      WHEN SUM(in_N) >= 30 THEN ROUND(100.0 * (SUM(in_N) - SUM(in_K)) / SUM(in_N), 1)
      ELSE NULL
    END AS sgi_30_display
  FROM t2
  GROUP BY borough
),
city AS (
  SELECT
    100.0 * SUM(n_gap_30d) / NULLIF(SUM(n_mature), 0) AS citywide_sgi_30
  FROM agg
)
SELECT
  a.borough,
  a.n_311,
  a.n_mature,
  a.n_timely_close_30d,
  a.n_gap_30d,
  a.n_gap_still_open,
  a.n_gap_late_closed,
  CASE WHEN a.n_gap_30d > 0 THEN ROUND(100.0 * a.n_gap_still_open / a.n_gap_30d, 1) END AS pct_gap_still_open,
  CASE WHEN a.n_gap_30d > 0 THEN ROUND(100.0 * a.n_gap_late_closed / a.n_gap_30d, 1) END AS pct_gap_late_closed,
  a.n_k_instant,
  CASE WHEN a.n_timely_close_30d > 0 THEN ROUND(100.0 * a.n_k_instant / a.n_timely_close_30d, 1) END AS pct_same_timestamp_close,
  a.sgi_30,
  a.sgi_30_display,
  a.sgi_30 - c.citywide_sgi_30 AS sgi_vs_city,
  ROUND(a.sgi_30 - c.citywide_sgi_30, 1) AS sgi_vs_city_display,
  current_timestamp() AS _built_at
FROM agg a
CROSS JOIN city c;

COMMENT ON TABLE workspace.default.gold_borough_index IS
  'Pooled SGI-30 by borough from tickets, not AVG of ZIP scores. Bronx 0.8 is close-stamp culture, not best-served. Staten Island 12.2 is not uniquely infested. Includes all mature tickets in that borough (even ZIPs suppressed on the ZIP leaderboard). Thin/small-N ZIP cards shrink toward this number. Still a paperwork gap, not a visit rate. Never average ZIP SGI values to get a borough or city number.';
COMMENT ON COLUMN workspace.default.gold_borough_index.borough IS
  'BRONX, BROOKLYN, MANHATTAN, QUEENS, or STATEN ISLAND from the 311 ticket.';
COMMENT ON COLUMN workspace.default.gold_borough_index.n_311 IS
  '311 rodent tickets with this borough.';
COMMENT ON COLUMN workspace.default.gold_borough_index.n_mature IS
  'Mature eligible tickets (N). Includes ZIPs that are not on the ZIP leaderboard.';
COMMENT ON COLUMN workspace.default.gold_borough_index.n_timely_close_30d IS
  'K: timely recorded closures. Administrative clock, not a visit.';
COMMENT ON COLUMN workspace.default.gold_borough_index.n_gap_30d IS
  'N minus K.';
COMMENT ON COLUMN workspace.default.gold_borough_index.n_gap_still_open IS
  'Mature tickets with no recorded close. Part of the gap.';
COMMENT ON COLUMN workspace.default.gold_borough_index.n_gap_late_closed IS
  'Mature tickets closed after 30 days. Part of the gap, not a visit.';
COMMENT ON COLUMN workspace.default.gold_borough_index.pct_gap_still_open IS
  'Share of borough gap that is still open.';
COMMENT ON COLUMN workspace.default.gold_borough_index.pct_gap_late_closed IS
  'Share of borough gap that closed late.';
COMMENT ON COLUMN workspace.default.gold_borough_index.n_k_instant IS
  'On-time K that is a same-timestamp administrative stamp. Still K; not an inspection.';
COMMENT ON COLUMN workspace.default.gold_borough_index.pct_same_timestamp_close IS
  'Percent of K that is an instant stamp. Why Closed% is not SGI.';
COMMENT ON COLUMN workspace.default.gold_borough_index.sgi_30 IS
  '100 * SUM(N-K) / SUM(N). Do not average ZIP SGI values to get this.';
COMMENT ON COLUMN workspace.default.gold_borough_index.sgi_30_display IS
  'Borough SGI-30 rounded to 1 decimal.';
COMMENT ON COLUMN workspace.default.gold_borough_index.sgi_vs_city IS
  'Borough pooled SGI minus city pooled SGI. Positive = larger paperwork gap than city.';
COMMENT ON COLUMN workspace.default.gold_borough_index.sgi_vs_city_display IS
  'sgi_vs_city rounded to 1 decimal.';
COMMENT ON COLUMN workspace.default.gold_borough_index._built_at IS
  'When this table was built.';

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_zip_index AS
WITH params AS (
  SELECT
    MAX(created_ts) AS observation_cutoff
  FROM workspace.default.silver_rat_sightings
),
t AS (
  SELECT
    r.unique_key,
    r.zip,
    r.descriptor_category,
    r.location_category,
    r.location_type,
    r.status,
    r.created_ts,
    r.closed_ts,
    r.is_same_timestamp_close,
    p.observation_cutoff,
    CASE WHEN r.dq_closed_flag IN ('parse_fail', 'negative_duration') THEN TRUE ELSE FALSE END AS is_outcome_excl,
    CASE
      WHEN r.created_ts <= p.observation_cutoff - INTERVAL 30 DAYS THEN TRUE
      ELSE FALSE
    END AS is_mature,
    CASE
      WHEN r.created_ts > p.observation_cutoff - INTERVAL 30 DAYS THEN TRUE
      ELSE FALSE
    END AS is_younger_30d,
    CASE
      WHEN r.status = 'In Progress'
           AND r.closed_ts IS NULL
           AND r.created_ts <= p.observation_cutoff - INTERVAL 14 DAYS THEN TRUE
      ELSE FALSE
    END AS is_backlog_14d
  FROM workspace.default.silver_rat_sightings r
  CROSS JOIN params p
  WHERE r.zip IS NOT NULL
    AND r.created_ts IS NOT NULL
    AND r.complaint_type = 'Rodent'
),
t2 AS (
  SELECT
    unique_key, zip, descriptor_category, location_category, location_type, status,
    is_same_timestamp_close, is_outcome_excl, is_mature, is_younger_30d, is_backlog_14d,
    closed_ts, created_ts, observation_cutoff,
    CASE WHEN is_mature AND NOT is_outcome_excl THEN TRUE ELSE FALSE END AS in_N,
    CASE
      WHEN is_mature AND NOT is_outcome_excl
           AND closed_ts IS NOT NULL
           AND closed_ts >= created_ts
           AND closed_ts <= created_ts + INTERVAL 30 DAYS
           AND closed_ts <= observation_cutoff THEN TRUE
      ELSE FALSE
    END AS in_K
  FROM t
),
t3 AS (
  SELECT
    t2.*,
    CASE WHEN in_N AND NOT in_K AND closed_ts IS NULL THEN TRUE ELSE FALSE END AS is_gap_still_open,
    CASE WHEN in_N AND NOT in_K AND closed_ts IS NOT NULL THEN TRUE ELSE FALSE END AS is_gap_late_closed,
    CASE WHEN in_K AND is_same_timestamp_close THEN TRUE ELSE FALSE END AS is_k_instant
  FROM t2
),
zip_311 AS (
  SELECT
    zip,
    COUNT(*) AS n_311,
    SUM(CASE WHEN in_N THEN 1 ELSE 0 END) AS n_mature,
    SUM(CASE WHEN in_K THEN 1 ELSE 0 END) AS n_timely_close_30d,
    SUM(CASE WHEN in_N AND NOT in_K THEN 1 ELSE 0 END) AS n_gap_30d,
    SUM(CASE WHEN is_gap_still_open THEN 1 ELSE 0 END) AS n_gap_still_open,
    SUM(CASE WHEN is_gap_late_closed THEN 1 ELSE 0 END) AS n_gap_late_closed,
    SUM(CASE WHEN is_k_instant THEN 1 ELSE 0 END) AS n_k_instant,
    SUM(CASE WHEN is_younger_30d THEN 1 ELSE 0 END) AS n_younger_than_30d,
    SUM(CASE WHEN is_outcome_excl THEN 1 ELSE 0 END) AS n_outcome_excluded,
    SUM(CASE WHEN is_backlog_14d THEN 1 ELSE 0 END) AS n_backlog_14d,
    SUM(CASE WHEN status = 'In Progress' THEN 1 ELSE 0 END) AS n_in_progress,
    SUM(CASE WHEN is_same_timestamp_close THEN 1 ELSE 0 END) AS n_same_timestamp_close,
    SUM(CASE WHEN descriptor_category = 'rat_sighting' THEN 1 ELSE 0 END) AS n_rat_sighting,
    SUM(CASE WHEN descriptor_category = 'mouse_sighting' THEN 1 ELSE 0 END) AS n_mouse_sighting,
    SUM(CASE WHEN descriptor_category = 'signs_of_rodents' THEN 1 ELSE 0 END) AS n_signs_of_rodents,
    SUM(CASE WHEN descriptor_category = 'condition_attracting_rodents' THEN 1 ELSE 0 END) AS n_condition_attracting,
    SUM(CASE WHEN location_category = 'dwelling' THEN 1 ELSE 0 END) AS n_dwelling,
    SUM(CASE WHEN location_type IN ('3+ Family Apt. Building', '3+ Family Mixed Use Building') THEN 1 ELSE 0 END) AS n_3plus_family,
    SUM(CASE WHEN location_type IN ('1-2 Family Dwelling', '1-2 Family Mixed Use Building') THEN 1 ELSE 0 END) AS n_1_2_family,
    SUM(CASE WHEN location_type = 'Single Room Occupancy (SRO)' THEN 1 ELSE 0 END) AS n_sro,
    SUM(CASE WHEN location_category = 'public_space' THEN 1 ELSE 0 END) AS n_public_space,
    SUM(CASE WHEN location_category = 'commercial' THEN 1 ELSE 0 END) AS n_commercial,
    SUM(CASE WHEN location_category = 'infrastructure' THEN 1 ELSE 0 END) AS n_infrastructure,
    SUM(CASE WHEN location_category = 'institution' THEN 1 ELSE 0 END) AS n_institution,
    SUM(CASE WHEN location_category = 'vacant' THEN 1 ELSE 0 END) AS n_vacant,
    SUM(CASE WHEN location_category = 'other' THEN 1 ELSE 0 END) AS n_other_location
  FROM t3
  GROUP BY zip
),
zip_kit AS (
  SELECT
    v.zip,
    COUNT(*) AS n_visits,
    COUNT(DISTINCT v.camis) AS n_restaurants,
    COUNT(DISTINCT CASE WHEN v.n_04k_rats > 0 THEN v.camis END) AS n_restaurants_04k,
    COUNT(DISTINCT CASE WHEN v.n_04l_mice > 0 THEN v.camis END) AS n_restaurants_04l,
    COUNT(DISTINCT CASE WHEN v.has_rodent_violation = 1 THEN v.camis END) AS n_restaurants_rodent,
    COUNT(DISTINCT CASE WHEN v.has_08a_harborage = 1 THEN v.camis END) AS n_restaurants_08a,
    SUM(v.has_rodent_violation) AS n_visits_rodent,
    SUM(v.has_08a_harborage) AS n_visits_08a,
    SUM(v.has_blank_violation_code) AS n_visits_blank_code
  FROM workspace.default.silver_inspection_visits v
  CROSS JOIN params p
  WHERE v.zip IS NOT NULL
    AND v.inspection_dt >= DATE '2025-01-02'
    AND v.inspection_dt < DATE '2026-09-17'
  GROUP BY v.zip
),
joined AS (
  SELECT
    s.zip,
    s.borough,
    s.borough_mode,
    s.borough_ambiguous,
    CASE WHEN s.zip = '10128' THEN 'neighborhood' ELSE s.zip_type END AS zip_type,
    s.max_cluster_n,
    s.n_distinct_clusters,
    s.max_cluster_share,
    s.is_cluster_dominant,
    CASE WHEN s.borough IS NULL THEN NULL ELSE b.sgi_30 END AS borough_sgi_30,
    CASE WHEN s.borough IS NULL THEN NULL ELSE b.sgi_30_display END AS borough_sgi_30_display,
    COALESCE(z.n_311, 0) AS n_311,
    COALESCE(z.n_mature, 0) AS n_mature,
    COALESCE(z.n_timely_close_30d, 0) AS n_timely_close_30d,
    COALESCE(z.n_gap_30d, 0) AS n_gap_30d,
    COALESCE(z.n_gap_still_open, 0) AS n_gap_still_open,
    COALESCE(z.n_gap_late_closed, 0) AS n_gap_late_closed,
    COALESCE(z.n_k_instant, 0) AS n_k_instant,
    COALESCE(z.n_younger_than_30d, 0) AS n_younger_than_30d,
    COALESCE(z.n_outcome_excluded, 0) AS n_outcome_excluded,
    COALESCE(z.n_backlog_14d, 0) AS n_backlog_14d,
    COALESCE(z.n_in_progress, 0) AS n_in_progress,
    COALESCE(z.n_same_timestamp_close, 0) AS n_same_timestamp_close,
    COALESCE(z.n_rat_sighting, 0) AS n_rat_sighting,
    COALESCE(z.n_mouse_sighting, 0) AS n_mouse_sighting,
    COALESCE(z.n_signs_of_rodents, 0) AS n_signs_of_rodents,
    COALESCE(z.n_condition_attracting, 0) AS n_condition_attracting,
    COALESCE(z.n_dwelling, 0) AS n_dwelling,
    COALESCE(z.n_3plus_family, 0) AS n_3plus_family,
    COALESCE(z.n_1_2_family, 0) AS n_1_2_family,
    COALESCE(z.n_sro, 0) AS n_sro,
    COALESCE(z.n_public_space, 0) AS n_public_space,
    COALESCE(z.n_commercial, 0) AS n_commercial,
    COALESCE(z.n_infrastructure, 0) AS n_infrastructure,
    COALESCE(z.n_institution, 0) AS n_institution,
    COALESCE(z.n_vacant, 0) AS n_vacant,
    COALESCE(z.n_other_location, 0) AS n_other_location,
    COALESCE(k.n_visits, 0) AS n_visits,
    COALESCE(k.n_restaurants, 0) AS n_restaurants,
    COALESCE(k.n_restaurants_04k, 0) AS n_restaurants_04k,
    COALESCE(k.n_restaurants_04l, 0) AS n_restaurants_04l,
    COALESCE(k.n_restaurants_rodent, 0) AS n_restaurants_rodent,
    COALESCE(k.n_restaurants_08a, 0) AS n_restaurants_08a,
    COALESCE(k.n_visits_rodent, 0) AS n_visits_rodent,
    COALESCE(k.n_visits_08a, 0) AS n_visits_08a,
    COALESCE(k.n_visits_blank_code, 0) AS n_visits_blank_code
  FROM workspace.default.silver_zip_spine s
  LEFT JOIN zip_311 z ON s.zip = z.zip
  LEFT JOIN zip_kit k ON s.zip = k.zip
  LEFT JOIN workspace.default.gold_borough_index b ON s.borough = b.borough
),
scored AS (
  SELECT
    j.*,
    CASE WHEN j.n_gap_30d > 0 THEN ROUND(100.0 * j.n_gap_still_open / j.n_gap_30d, 1) END AS pct_gap_still_open,
    CASE WHEN j.n_gap_30d > 0 THEN ROUND(100.0 * j.n_gap_late_closed / j.n_gap_30d, 1) END AS pct_gap_late_closed,
    CASE WHEN j.n_timely_close_30d > 0 THEN ROUND(100.0 * j.n_k_instant / j.n_timely_close_30d, 1) END AS pct_same_timestamp_close,
    CASE
      WHEN n_mature >= 30 THEN 100.0 * n_gap_30d / n_mature
      ELSE NULL
    END AS sgi_30,
    CASE
      WHEN n_mature >= 30 THEN ROUND(100.0 * n_gap_30d / n_mature, 1)
      ELSE NULL
    END AS sgi_30_display,
    CASE
      WHEN n_restaurants > 0 THEN ROUND(100.0 * n_restaurants_04k / n_restaurants, 1)
      ELSE NULL
    END AS pct_establishments_04k,
    CASE
      WHEN n_restaurants > 0 THEN ROUND(100.0 * n_restaurants_04l / n_restaurants, 1)
      ELSE NULL
    END AS pct_establishments_04l,
    CASE
      WHEN n_restaurants > 0 THEN ROUND(100.0 * n_restaurants_rodent / n_restaurants, 1)
      ELSE NULL
    END AS pct_establishments_rodent,
    CASE
      WHEN n_restaurants > 0 THEN ROUND(100.0 * n_restaurants_08a / n_restaurants, 1)
      ELSE NULL
    END AS pct_establishments_08a,
    CASE
      WHEN n_restaurants > 0 THEN ROUND(n_311 / n_restaurants, 2)
      ELSE NULL
    END AS complaints_per_observed_establishment,
    CASE
      WHEN n_311 = 0 THEN 'no_311'
      WHEN n_mature = 0 THEN 'no_mature_tickets'
      WHEN n_mature < 30 THEN 'insufficient_data'
      WHEN zip_type = 'neighborhood' THEN 'published_ranked'
      ELSE 'published_not_ranked'
    END AS sgi_status,
    CASE
      WHEN zip_type = 'neighborhood' AND n_mature >= 30 THEN TRUE
      ELSE FALSE
    END AS is_rank_eligible,
    CASE WHEN n_mature < 100 THEN TRUE ELSE FALSE END AS is_small_n,
    CASE WHEN zip IN ('10001', '10018', '10019', '10036', '10020') THEN TRUE ELSE FALSE END AS is_office_tourist_zip,
    CASE
      WHEN zip IN ('11430', '11371') THEN 'airport'
      WHEN zip LIKE '101%' AND zip <> '10128' THEN 'building'
      WHEN zip_type = 'non_nyc' THEN 'non_nyc'
      WHEN zip IN ('10001', '10018', '10019', '10036', '10020') THEN 'office_tourist'
      WHEN n_311 > 0 AND n_institution * 1.0 / n_311 >= 0.25 THEN 'institution'
      WHEN n_311 > 0 AND n_dwelling * 1.0 / n_311 >= 0.70 THEN 'residential'
      WHEN n_311 > 0 AND (n_commercial + n_public_space + n_infrastructure) * 1.0 / n_311 >= 0.40 THEN 'street_commercial'
      ELSE 'other'
    END AS zip_character,
    CASE WHEN zip = '10035' THEN TRUE ELSE FALSE END AS is_stress_test_zip,
    CASE WHEN n_311 = 0 THEN TRUE ELSE FALSE END AS has_no_311,
    CASE WHEN n_restaurants = 0 THEN TRUE ELSE FALSE END AS has_no_kitchens
  FROM joined j
),
ranked AS (
  SELECT
    s.*,
    RANK() OVER (
      ORDER BY CASE WHEN is_rank_eligible THEN sgi_30 END DESC NULLS LAST,
               CASE WHEN is_rank_eligible THEN n_mature END DESC NULLS LAST,
               zip
    ) AS rank_city_raw,
    RANK() OVER (
      PARTITION BY borough
      ORDER BY CASE WHEN is_rank_eligible THEN sgi_30 END DESC NULLS LAST,
               CASE WHEN is_rank_eligible THEN n_mature END DESC NULLS LAST,
               zip
    ) AS rank_boro_raw
  FROM scored s
)
SELECT
  zip,
  borough,
  borough_mode,
  borough_ambiguous,
  zip_type,
  zip_character,
  is_office_tourist_zip,
  n_311,
  n_mature,
  n_timely_close_30d,
  n_gap_30d,
  n_gap_still_open,
  n_gap_late_closed,
  pct_gap_still_open,
  pct_gap_late_closed,
  n_k_instant,
  pct_same_timestamp_close,
  n_younger_than_30d,
  n_outcome_excluded,
  sgi_30,
  sgi_30_display,
  sgi_status,
  is_rank_eligible,
  is_small_n,
  borough_sgi_30,
  borough_sgi_30_display,
  CASE WHEN is_rank_eligible THEN rank_city_raw END AS rank_sgi_citywide,
  CASE WHEN is_rank_eligible AND borough IS NOT NULL THEN rank_boro_raw END AS rank_sgi_borough,
  n_backlog_14d,
  n_in_progress,
  n_same_timestamp_close,
  n_rat_sighting,
  n_mouse_sighting,
  n_signs_of_rodents,
  n_condition_attracting,
  n_dwelling,
  n_3plus_family,
  n_1_2_family,
  n_sro,
  n_public_space,
  n_commercial,
  n_infrastructure,
  n_institution,
  n_vacant,
  n_other_location,
  n_distinct_clusters,
  max_cluster_n,
  max_cluster_share,
  is_cluster_dominant,
  n_visits,
  n_restaurants,
  n_restaurants_04k,
  n_restaurants_04l,
  n_restaurants_rodent,
  n_restaurants_08a,
  n_visits_rodent,
  n_visits_08a,
  n_visits_blank_code,
  pct_establishments_04k,
  pct_establishments_04l,
  pct_establishments_rodent,
  pct_establishments_08a,
  complaints_per_observed_establishment,
  is_stress_test_zip,
  has_no_311,
  has_no_kitchens,
  CONCAT_WS(
    ' ',
    'SGI-30 is a 30-day paperwork clock, not proof the city showed up. Instant administrative stamps (closed_ts = created_ts) still count as K; they are not inspections. Do not rebuild SGI from Closed%.',
    '311 is who still files, not a rat census. This packet has no trust, tenure, language, HPD, population, visit log, or inspection_type. Quiet is not clean. Ranking by 311 sends staff where attention already is. Indoor mice are often HPD, not this DOHMH Rodent file. This is the ZIP, not your building or block. Do not fetch ACS.',
    CASE WHEN zip_type = 'airport' THEN 'Airport ZIP. Kitchen coverage only. Do not rank SGI. Few calls does not mean good service.' END,
    CASE WHEN zip_type = 'building' THEN 'This ZIP is a building (101xx), not a neighborhood block.' END,
    CASE WHEN zip = '10128' THEN '10128 is Upper East Side neighborhood housing, not a 101xx building ZIP.' END,
    CASE WHEN zip_type = 'non_nyc' THEN 'Not an NYC neighborhood in this packet. Lookup only. Out of scope for ranking. Do not impute. Do not call it clean.' END,
    CASE WHEN zip_type = 'thin' OR sgi_status = 'insufficient_data' THEN 'Too few mature tickets for a published SGI. Show raw counts plus borough_sgi_30. Do not invent 0 or 100.' END,
    CASE WHEN is_small_n AND n_mature >= 30 THEN 'Small mature N (under 100). SGI is unstable. Shrink toward borough_sgi_30; never AVG of ZIP scores.' END,
    CASE WHEN borough IN ('STATEN ISLAND', 'QUEENS', 'BRONX') THEN 'Borough close-stamp practice differs (SI/Queens stamp slower than Bronx). Compare to borough_sgi_30, not this borough is fine.' END,
    CASE WHEN borough_ambiguous THEN 'Split ZIP (10463 Marble Hill or 11370 Rikers/East Elmhurst). Borough is unresolved. No borough SGI assigned.' END,
    CASE WHEN has_no_311 THEN 'No recorded 311 rodent tickets here. That is not zero rats. People may not call.' END,
    CASE WHEN has_no_kitchens AND n_311 > 0 THEN 'No restaurants in the 2025-now inspection file. Restaurant rates are undefined, not zero. Of restaurants in 2025-now inspections, not all food on the block.' END,
    CASE WHEN is_cluster_dominant THEN 'One GPS cluster is over 30% of tickets. Say one site, not cursed ZIP. Not a lot (no BBL). Cluster is a flag, not zip_character.' END,
    CASE WHEN zip IN ('11103', '11366', '10038', '10024', '10452') THEN 'Cluster-dominant ZIP. One GPS pile-up can inflate 311. Do not crown it cursed or best-served.' END,
    CASE WHEN is_stress_test_zip THEN 'East Harlem stress test. High 311 can mean outreach (rat mitigation), not a unique infestation. Tiny SGI does not mean best served. Character can still be residential.' END,
    CASE WHEN is_office_tourist_zip THEN 'Office/tourist ZIP (Midtown list). Low 311 vs many kitchens is land use, not virtue. office_tourist is not hour-of-day.' END,
    CASE WHEN zip_character = 'institution' THEN 'Many tickets are school/hospital/daycare, not proof this ZIP is a campus.' END,
    CASE WHEN zip_character = 'residential' THEN 'Most 311 here is dwellings (homes/apts). Still not a rat census.' END,
    CASE WHEN zip_character = 'street_commercial' THEN 'Many tickets are street/storefront/sewer, not bedrooms. Do not average a sewer and a bedroom.' END,
    CASE WHEN zip_character = 'other' THEN 'No dominant 311 location mix in this extract. zip_character other is not location_category other.' END,
    CASE WHEN n_same_timestamp_close > 0 THEN 'Some tickets close at the same timestamp they were opened. Closed is a clock, not proof of a visit.' END,
    'Kitchen 04K/04L/08A are a parallel DOHMH program, not a 311 response. Tuesday inspection is not Monday 311. Never AVG(sgi_30). High SGI means review administrative handling, not neglect or infestation.'
  ) AS how_to_read,
  (SELECT observation_cutoff FROM params) AS observation_cutoff,
  current_timestamp() AS _built_at
FROM ranked;

COMMENT ON TABLE workspace.default.gold_zip_index IS
  'ONE ROW PER ZIP. Service Gap Index (SGI-30) plus 311 demand and kitchen context. SGI-30 is the percent of mature rodent 311 tickets with no recorded closure within 30 days — a paperwork gap, NOT proof of a visit, NOT a rat census, NOT Closed percent of all tickets. Instant stamps still count as K; see n_k_instant. Rank only where is_rank_eligible is true (zip_type neighborhood AND mature N >= 30). zip_character is a packet-only card label, not the rank gate and not census land use. Higher SGI = larger gap. 10035 is a stress-test ZIP (one GPS cluster + outreach), not the villain. Airports 11430/11371: kitchen coverage only. 10128 is neighborhood (UES), not a building. Sources were cleaned separately then joined on ZIP; a kitchen inspection is not the response to a 311 call. Prefer this table for ZIP lookup. Observation cutoff is MAX(created_ts) in the extract, not now(). Genie: ranks are precomputed; filter after ranking. Never AVG(sgi_30).';

num_affected_rows,num_inserted_rows


In [0]:
%sql
COMMENT ON COLUMN workspace.default.gold_zip_index.zip IS
  'ZIP code, not a block or neighborhood name. Observed in 311 or kitchens. Missing ZIP in a lookup means not in this extract.';
COMMENT ON COLUMN workspace.default.gold_zip_index.borough IS
  'NYC borough when the packet has exactly one borough for this ZIP. NULL if ambiguous (10463, 11370). No borough SGI when NULL.';
COMMENT ON COLUMN workspace.default.gold_zip_index.borough_mode IS
  'Most common borough even if ambiguous. Do not treat as resolved when borough_ambiguous is true.';
COMMENT ON COLUMN workspace.default.gold_zip_index.borough_ambiguous IS
  'TRUE for 10463 (Marble Hill) and 11370 (Rikers / East Elmhurst).';
COMMENT ON COLUMN workspace.default.gold_zip_index.zip_type IS
  'Rank gate: neighborhood / airport (11430 JFK, 11371 LGA) / building (101xx except 10128) / non_nyc / thin (n_311 < 10). neighborhood here means ordinary NYC ZIP eligible if N is large enough, not land use. Only neighborhood can be SGI-ranked. 10128 is neighborhood.';
COMMENT ON COLUMN workspace.default.gold_zip_index.zip_character IS
  'Packet-only card label, first match: airport, building, non_nyc, office_tourist, institution, residential, street_commercial, other. NOT census land use. NOT campus. NOT the rank gate. leftover other = no dominant mix, not location_category other. Thin volume and GPS clusters are flags, not character values.';
COMMENT ON COLUMN workspace.default.gold_zip_index.is_office_tourist_zip IS
  'TRUE for hardcoded Midtown list 10001/10018/10019/10036/10020. Not hour-of-day.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_311 IS
  'Distinct 311 rodent tickets in this ZIP. Demand among people who called, not a rat census. Zero is not zero rats. All four descriptors. Never call this rat sightings.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_mature IS
  'N: tickets created at least 30 days before the observation cutoff, after quality exclusions. Denominator of SGI-30. Younger tickets are not in N.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_timely_close_30d IS
  'K: mature tickets with a recorded closure within 30 days of created_ts and by the cutoff. Administrative clock, not a visit. Includes instant stamps.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_gap_30d IS
  'N minus K. Mature tickets without a timely recorded closure. Equals n_gap_still_open + n_gap_late_closed.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_gap_still_open IS
  'Mature gap tickets with no recorded close. Not delayed service on tickets filed yesterday (those are n_younger_than_30d).';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_gap_late_closed IS
  'Mature gap tickets that did close, but after 30 days. Still not a visit.';
COMMENT ON COLUMN workspace.default.gold_zip_index.pct_gap_still_open IS
  'Percent of n_gap_30d that is still open. NULL if no gap.';
COMMENT ON COLUMN workspace.default.gold_zip_index.pct_gap_late_closed IS
  'Percent of n_gap_30d that closed late. NULL if no gap.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_k_instant IS
  'Count of K that is a same-timestamp administrative stamp. Still K in SGI-30. Do not drop from K. Do not call them inspections. Packet cannot name the operational cause.';
COMMENT ON COLUMN workspace.default.gold_zip_index.pct_same_timestamp_close IS
  'Percent of K that is an instant stamp. Citywide about half. Why Closed percent must not be SGI.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_younger_than_30d IS
  'Tickets created in the last 30 days of the extract. Excluded from SGI. Do not treat open young tickets as delayed service.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_outcome_excluded IS
  'Tickets with unusable closure clocks (parse fail or closed before created). Counted here, not in N or K.';
COMMENT ON COLUMN workspace.default.gold_zip_index.sgi_30 IS
  '100 * (N-K) / N when N >= 30, else NULL. Scale 0-100. Higher = larger 30-day administrative closure gap. Never 0 or 100 when N = 0. Do not call this a show-up rate.';
COMMENT ON COLUMN workspace.default.gold_zip_index.sgi_30_display IS
  'SGI-30 rounded to 1 decimal for display. NULL when N < 30. Use sgi_30 for ordering.';
COMMENT ON COLUMN workspace.default.gold_zip_index.sgi_status IS
  'published_ranked / published_not_ranked / insufficient_data / no_mature_tickets / no_311. Only published_ranked belongs on a leaderboard.';
COMMENT ON COLUMN workspace.default.gold_zip_index.is_rank_eligible IS
  'TRUE only for zip_type = neighborhood AND N >= 30. Filter to this before top-N. Always compute citywide ranks on the full table, then filter.';
COMMENT ON COLUMN workspace.default.gold_zip_index.is_small_n IS
  'TRUE when n_mature < 100. Published SGI can still exist if N >= 30, but it is unstable. Shrink toward borough_sgi_30.';
COMMENT ON COLUMN workspace.default.gold_zip_index.borough_sgi_30 IS
  'Pooled borough SGI from gold_borough_index. NULL if borough is NULL (10463, 11370). Thin/small-N cards shrink toward this, never toward AVG of ZIP scores.';
COMMENT ON COLUMN workspace.default.gold_zip_index.borough_sgi_30_display IS
  'borough_sgi_30 rounded to 1 decimal.';
COMMENT ON COLUMN workspace.default.gold_zip_index.rank_sgi_citywide IS
  '1 = largest SGI-30 among rank-eligible ZIPs (biggest paperwork gap), not most rats and not proven worst service. NULL if not rank-eligible. Ties share a rank.';
COMMENT ON COLUMN workspace.default.gold_zip_index.rank_sgi_borough IS
  'Rank within borough among rank-eligible ZIPs. NULL if not eligible or borough is NULL.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_backlog_14d IS
  'In Progress tickets created more than 14 days before the cutoff. Better still-open display than open-rate on tickets filed yesterday.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_in_progress IS
  'Current In Progress count. Includes young tickets. Do not use as SGI.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_same_timestamp_close IS
  'All tickets with closed_ts equal to created_ts, not only K. A fast close may be a visit, duplicate, no access, bounce, or batch stamp. Packet cannot name the cause.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_rat_sighting IS
  'Tickets with descriptor Rat Sighting. Indoor mice are not this column.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_mouse_sighting IS
  'Mouse Sighting tickets. Do not match these to 04K kitchen rat evidence.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_signs_of_rodents IS
  'Signs of Rodents. Species unknown.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_condition_attracting IS
  'Condition Attracting Rodents. Not a confirmed animal.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_dwelling IS
  'Tickets in homes/apartments (~77% citywide). Different job from sewers or lots.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_3plus_family IS
  '3+ family apartment or mixed-use. Density / landlord-fear hint, not NYCHA proof.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_1_2_family IS
  '1-2 family dwelling or mixed-use.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_sro IS
  'Single Room Occupancy tickets. Subset of dwelling. Needed so dwelling pieces plus other exclusive location counts can be audited.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_public_space IS
  'Sidewalk, street, stairs, public garden.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_commercial IS
  'Commercial building, office, parking lot. Tourist/office ZIPs can inflate this.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_infrastructure IS
  'Catch basin/sewer or construction site. Do not average with bedrooms.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_institution IS
  'School, day care, hospital, government building, summer camp. Must be in the location split so counts sum to n_311. Not a campus ZIP.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_vacant IS
  'Vacant lot or vacant building.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_other_location IS
  'Location type Other. Exclusive bucket. Not zip_character other.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_distinct_clusters IS
  'Unique GPS points (rounded lat/long). Density proxy because population is not in the packet.';
COMMENT ON COLUMN workspace.default.gold_zip_index.max_cluster_n IS
  'Tickets at the busiest GPS cluster. 10035 is about 1227 at one point.';
COMMENT ON COLUMN workspace.default.gold_zip_index.max_cluster_share IS
  'Share of tickets at the busiest GPS cluster. 10035 ~0.817. One site, not 1501 independent rats.';
COMMENT ON COLUMN workspace.default.gold_zip_index.is_cluster_dominant IS
  'TRUE only if n_311 >= 30 AND one cluster is more than 30% of tickets. Then say one site, not cursed ZIP. Flag, not zip_character. Fires for 10035, 11103, 11366, 10038, 10024, 10452.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_visits IS
  'Restaurant establishment-date proxies in the shared window (2025-01-02 through 2026-09-16). Not exact inspection count. Places DOHMH already visits, not all food on the block. Boundary-day completeness is unverified.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_restaurants IS
  'COUNT DISTINCT camis. Denominator because population is not in the packet. Punishes residential ZIPs vs kitchen-dense ones. 0 means rates are undefined, not zero.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_restaurants_04k IS
  'Distinct establishments with any 04K (rat evidence) in the window.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_restaurants_04l IS
  'Distinct establishments with any 04L (mouse evidence) in the window.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_restaurants_rodent IS
  'Distinct establishments with 04K or 04L. Union, not a sum of the two columns (overlap exists).';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_restaurants_08a IS
  'Distinct establishments with 08A harborage. Not a live rat. Keep separate from rodent evidence.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_visits_rodent IS
  'Visits with 04K or 04L. Parallel kitchen program, not a 311 response.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_visits_08a IS
  'Visits with 08A harborage.';
COMMENT ON COLUMN workspace.default.gold_zip_index.n_visits_blank_code IS
  'Visits with a blank violation code. Unknown finding, not a clean pass.';
COMMENT ON COLUMN workspace.default.gold_zip_index.pct_establishments_04k IS
  'Percent of observed establishments with a recorded 04K. Not citywide infestation and not all restaurants. NULL if n_restaurants = 0.';
COMMENT ON COLUMN workspace.default.gold_zip_index.pct_establishments_04l IS
  'Percent of observed establishments with 04L. NULL if no restaurants.';
COMMENT ON COLUMN workspace.default.gold_zip_index.pct_establishments_rodent IS
  'Percent with 04K or 04L. NULL if no restaurants.';
COMMENT ON COLUMN workspace.default.gold_zip_index.pct_establishments_08a IS
  'Percent with 08A harborage. Not live-rat prevalence. NULL if no restaurants.';
COMMENT ON COLUMN workspace.default.gold_zip_index.complaints_per_observed_establishment IS
  'n_311 / n_restaurants. Exploratory land-use ratio, NOT SGI and NOT people-adjusted demand. 10035 looks extreme because it has few kitchens. NULL if no restaurants.';
COMMENT ON COLUMN workspace.default.gold_zip_index.is_stress_test_zip IS
  'TRUE for 10035 East Harlem. Known outreach / rat-mitigation ZIP. Extra 311 can mean the city asked people to call.';
COMMENT ON COLUMN workspace.default.gold_zip_index.has_no_311 IS
  'TRUE if zero 311 tickets. Not proof the area is clean.';
COMMENT ON COLUMN workspace.default.gold_zip_index.has_no_kitchens IS
  'TRUE if zero observed establishments. Restaurant percentages must stay NULL.';
COMMENT ON COLUMN workspace.default.gold_zip_index.how_to_read IS
  'Plain-language caveats for this ZIP. Read this before ranking or calling a ZIP cursed. Names paperwork clock, who-calls holes, ZIP vs block, borough stamps, small-N, kitchens parallel, zip_character, cluster/volume.';
COMMENT ON COLUMN workspace.default.gold_zip_index.observation_cutoff IS
  'Inferred as-of time: MAX(created_ts) on 311. Not a verified refresh timestamp. Do not use current_timestamp to age this snapshot.';
COMMENT ON COLUMN workspace.default.gold_zip_index._built_at IS
  'When this gold table was built.';

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.gold_zip_leaderboard AS
SELECT *
FROM workspace.default.gold_zip_index
WHERE is_rank_eligible = TRUE;

COMMENT ON TABLE workspace.default.gold_zip_leaderboard IS
  'VIEW of gold_zip_index restricted to neighborhood ZIPs with mature N >= 30. Use for top-N SGI questions. Rank 1 is the largest 30-day administrative closure gap among N>=30, never worst rats. Airports, buildings, NJ, and thin ZIPs are excluded. For a user ZIP lookup, use gold_zip_index (so the ZIP does not falsely rank first). Never rank after filtering to one ZIP. Show N on every bar. Small-N (n_mature < 100) is unstable.';

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_zip_month AS
WITH params AS (
  SELECT MAX(created_ts) AS observation_cutoff
  FROM workspace.default.silver_rat_sightings
),
t AS (
  SELECT
    r.zip,
    DATE_TRUNC('MONTH', r.created_ts) AS created_month,
    r.created_ts,
    r.closed_ts,
    p.observation_cutoff,
    CASE WHEN r.dq_closed_flag IN ('parse_fail', 'negative_duration') THEN TRUE ELSE FALSE END AS is_outcome_excl
  FROM workspace.default.silver_rat_sightings r
  CROSS JOIN params p
  WHERE r.zip IS NOT NULL
    AND r.created_ts IS NOT NULL
    AND r.complaint_type = 'Rodent'
),
flagged AS (
  SELECT
    zip,
    created_month,
    observation_cutoff,
    CASE
      WHEN LAST_DAY(created_month) <= DATE(observation_cutoff) - INTERVAL 30 DAYS THEN TRUE
      ELSE FALSE
    END AS is_month_fully_mature,
    CASE
      WHEN created_ts <= observation_cutoff - INTERVAL 30 DAYS AND NOT is_outcome_excl THEN 1
      ELSE 0
    END AS in_N,
    CASE
      WHEN created_ts <= observation_cutoff - INTERVAL 30 DAYS
           AND NOT is_outcome_excl
           AND closed_ts IS NOT NULL
           AND closed_ts >= created_ts
           AND closed_ts <= created_ts + INTERVAL 30 DAYS
           AND closed_ts <= observation_cutoff THEN 1
      ELSE 0
    END AS in_K,
    1 AS n_created
  FROM t
)
SELECT
  zip,
  created_month,
  MAX(CASE WHEN is_month_fully_mature THEN 1 ELSE 0 END) = 1 AS is_month_fully_mature,
  SUM(n_created) AS n_created,
  SUM(in_N) AS n_mature,
  SUM(in_K) AS n_timely_close_30d,
  SUM(in_N) - SUM(in_K) AS n_gap_30d,
  CASE
    WHEN MAX(CASE WHEN is_month_fully_mature THEN 1 ELSE 0 END) = 1
         AND SUM(in_N) > 0
      THEN 100.0 * (SUM(in_N) - SUM(in_K)) / SUM(in_N)
    ELSE NULL
  END AS sgi_30,
  CASE
    WHEN MAX(CASE WHEN is_month_fully_mature THEN 1 ELSE 0 END) = 1
         AND SUM(in_N) > 0
      THEN ROUND(100.0 * (SUM(in_N) - SUM(in_K)) / SUM(in_N), 1)
    ELSE NULL
  END AS sgi_30_display,
  MAX(observation_cutoff) AS observation_cutoff,
  current_timestamp() AS _built_at
FROM flagged
GROUP BY zip, created_month;

COMMENT ON TABLE workspace.default.gold_zip_month IS
  'ZIP by calendar month of 311 created_ts. SGI-30 is computed ONLY for fully mature months (last day of month is at least 30 days before observation_cutoff). Incomplete months (almost all leftover In Progress is Aug-Sep 2026) have NULL SGI. FORBID averaging monthly SGI into an annual score. FORBID comparing one ZIP July to another ZIP January. FORBID summing monthly distinct camis (this table has no restaurants). Season triples volume; same window for ZIP ranking washes season. Optional citywide week is gold_citywide_week — do not overfit a ZIP-week model.';
COMMENT ON COLUMN workspace.default.gold_zip_month.zip IS
  'ZIP of the ticket. Same grain key as gold_zip_index.zip.';
COMMENT ON COLUMN workspace.default.gold_zip_month.created_month IS
  'First day of the created month.';
COMMENT ON COLUMN workspace.default.gold_zip_month.is_month_fully_mature IS
  'TRUE if every ticket in this month had 30 days to close by the cutoff. Exclude incomplete months from month-over-month.';
COMMENT ON COLUMN workspace.default.gold_zip_month.n_created IS
  '311 tickets created in this ZIP-month. Not rats.';
COMMENT ON COLUMN workspace.default.gold_zip_month.n_mature IS
  'Tickets from this month that are mature vs the cutoff. Equals n_created when the month is fully mature.';
COMMENT ON COLUMN workspace.default.gold_zip_month.n_timely_close_30d IS
  'K among mature tickets in this month.';
COMMENT ON COLUMN workspace.default.gold_zip_month.n_gap_30d IS
  'Mature N minus K for this month.';
COMMENT ON COLUMN workspace.default.gold_zip_month.sgi_30 IS
  'NULL unless the month is fully mature. Do not average these into a yearly ZIP SGI.';
COMMENT ON COLUMN workspace.default.gold_zip_month.sgi_30_display IS
  'Monthly SGI rounded, NULL if month incomplete.';
COMMENT ON COLUMN workspace.default.gold_zip_month.observation_cutoff IS
  'MAX(created_ts) on 311 for this gold build.';
COMMENT ON COLUMN workspace.default.gold_zip_month._built_at IS
  'When this table was built.';

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Rebuilt from silver so it inherits the NYC geography gate (222 ZIPs, not 234). Previously an orphan table
-- built at 16:25 that nothing refreshed; two dashboard datasets read it and leaked NJ/LI ZIPs into the filter.
CREATE OR REPLACE TABLE workspace.default.gold_zip_temporal AS
WITH params AS (
  SELECT MAX(created_ts) AS observation_cutoff
  FROM workspace.default.silver_rat_sightings
),
t AS (
  SELECT
    r.zip,
    r.created_ts,
    r.closed_ts,
    p.observation_cutoff,
    DATE_TRUNC('WEEK', r.created_ts) AS week_start,
    CASE WHEN MONTH(r.created_ts) BETWEEN 5 AND 9 THEN 1 ELSE 0 END AS is_warm,
    CASE
      WHEN r.created_ts <= p.observation_cutoff - INTERVAL 30 DAYS
           AND r.dq_closed_flag NOT IN ('parse_fail', 'negative_duration')
           AND r.closed_ts IS NOT NULL
           AND r.closed_ts >= r.created_ts
        THEN DATEDIFF(r.closed_ts, r.created_ts)
    END AS days_to_close
  FROM workspace.default.silver_rat_sightings r
  CROSS JOIN params p
  WHERE r.zip IS NOT NULL
    AND r.created_ts IS NOT NULL
    AND r.complaint_type = 'Rodent'
),
weeks AS (
  SELECT zip, week_start, COUNT(*) AS n_week
  FROM t
  GROUP BY zip, week_start
),
peak_agg AS (
  SELECT w.zip, MAX(w.n_week) AS peak_week_n, COUNT(*) AS n_weeks_with_tickets
  FROM weeks w GROUP BY w.zip
),
peak_start AS (
  SELECT zip, MIN(week_start) AS peak_week_start
  FROM weeks w
  WHERE n_week = (SELECT MAX(n_week) FROM weeks x WHERE x.zip = w.zip)
  GROUP BY zip
),
agg AS (
  SELECT
    zip,
    COUNT(*) AS n_311,
    ROUND(PERCENTILE(days_to_close, 0.5), 1) AS median_days_to_close,
    ROUND(PERCENTILE(days_to_close, 0.25), 1) AS p25_days_to_close,
    ROUND(PERCENTILE(days_to_close, 0.75), 1) AS p75_days_to_close,
    COUNT(days_to_close) AS n_closed_valid_duration,
    SUM(CASE WHEN days_to_close = 0 THEN 1 ELSE 0 END) AS n_zero_day_close,
    SUM(is_warm) AS n_warm_season,
    SUM(1 - is_warm) AS n_cool_season,
    MIN(created_ts) AS first_ticket_ts,
    MAX(created_ts) AS last_ticket_ts
  FROM t
  GROUP BY zip
)
SELECT
  s.zip,
  a.median_days_to_close,
  a.p25_days_to_close,
  a.p75_days_to_close,
  COALESCE(a.n_closed_valid_duration, 0) AS n_closed_valid_duration,
  COALESCE(a.n_zero_day_close, 0) AS n_zero_day_close,
  COALESCE(a.n_warm_season, 0) AS n_warm_season,
  COALESCE(a.n_cool_season, 0) AS n_cool_season,
  CASE WHEN a.n_311 > 0 THEN ROUND(100.0 * a.n_warm_season / a.n_311, 1) END AS warm_season_pct,
  COALESCE(pa.peak_week_n, 0) AS peak_week_n,
  ps.peak_week_start,
  CASE WHEN a.n_311 > 0 THEN ROUND(pa.peak_week_n / a.n_311, 3) END AS peak_week_share,
  COALESCE(pa.n_weeks_with_tickets, 0) AS n_weeks_with_tickets,
  a.first_ticket_ts,
  a.last_ticket_ts,
  CONCAT_WS(' ',
    CASE
      WHEN COALESCE(a.n_closed_valid_duration, 0) = 0
        THEN 'No valid-duration closed tickets. Median close time is undefined, not zero.'
      WHEN a.n_closed_valid_duration < 30
        THEN 'Fewer than 30 closed tickets with valid duration. Median is unstable — shrink toward borough.'
    END,
    CASE WHEN a.n_closed_valid_duration > 0 AND a.n_zero_day_close * 1.0 / a.n_closed_valid_duration > 0.30
      THEN 'Over 30pct of closed tickets closed in zero days. Instant closes are administrative clocks, not visits.' END,
    CASE WHEN a.n_311 > 0 AND pa.peak_week_n * 1.0 / a.n_311 > 0.15
      THEN 'One week accounts for over 15pct of this ZIPs tickets. Possible viral event or news spike, not a sustained change in rats or service.' END,
    'Median days to close is among closed tickets only — selection bias: open tickets are excluded. Do not treat as first-visit time. Seasonal split shows when people call, not when rats are active. The observation window is the same for every ZIP (Jan 2025 to now), so season is mostly a wash for ranking. Do not compare one ZIPs July to another ZIPs January.'
  ) AS how_to_read_temporal,
  (SELECT observation_cutoff FROM params) AS observation_cutoff,
  current_timestamp() AS _built_at
FROM workspace.default.silver_zip_spine s
LEFT JOIN agg a ON s.zip = a.zip
LEFT JOIN peak_agg pa ON s.zip = pa.zip
LEFT JOIN peak_start ps ON s.zip = ps.zip;

COMMENT ON TABLE workspace.default.gold_zip_temporal IS
  'ONE ROW PER ZIP on silver_zip_spine (NYC only; NJ / Long Island / Westchester removed upstream). Temporal supplements to gold_zip_index: median days to recorded closure, seasonal split (warm May-Sep / cool Oct-Apr), peak-week spike detection, and observation span. Join to gold_zip_index on zip. Median close is among closed tickets only (selection bias). Seasonal split shows when people call, not when rats are active. Peak-week share flags viral events or news spikes, not sustained infestation changes. Do not treat any column here as proof of a visit or first-response time. Kitchen-only ZIPs have zero counts and NULL medians.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.zip IS
  'ZIP code. Join key to gold_zip_index.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.median_days_to_close IS
  'Median DATEDIFF(closed_ts, created_ts) among mature closed tickets with valid nonnegative duration, rounded to 1 decimal day. Selection bias: open tickets are excluded, so this is NOT first-visit time. A fast median may mean instant administrative closes, not rapid visits.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.p25_days_to_close IS
  '25th percentile of days to close. Shows the floor of the distribution. Many ZIPs have 0 here (instant closes).';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.p75_days_to_close IS
  '75th percentile of days to close. Shows the tail. A large gap between median and p75 means a long tail of slow closures.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.n_closed_valid_duration IS
  'Count of closed tickets with valid nonnegative duration used for the median. This is the denominator for the median, not N or n_311. If < 30, median is unstable.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.n_zero_day_close IS
  'Closed tickets where DATEDIFF is 0 (same-day close). Tens of thousands citywide. Instant close is an administrative clock, not proof of a visit, duplicate, no access, or bounce.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.n_warm_season IS
  'Tickets created May through September. Rats are more visible in warm months. Not a rat activity count — it is a call count.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.n_cool_season IS
  'Tickets created October through April. Fewer calls does not mean fewer rats. Season is mostly a wash for ranking since the window is the same for every ZIP.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.warm_season_pct IS
  'Percent of tickets in warm months. A ZIP with 80pct warm-season calls may be park- or sidewalk-driven. Do not compare one ZIPs July to another ZIPs January.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.peak_week_n IS
  'Most tickets in any single week (DATE_TRUNC WEEK, Monday start) for this ZIP. Used for spike detection.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.peak_week_start IS
  'The Monday of the peak week. If several weeks tie for the peak, the earliest is shown (deterministic). If it aligns with a known news event or viral video, the spike is attention, not infestation.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.peak_week_share IS
  'Peak week tickets / total tickets. Over 0.15 suggests a spike (viral video, news week, or outreach push), not a sustained change in rats or service. Do not overfit to one week.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.n_weeks_with_tickets IS
  'Distinct weeks with at least one ticket. Breadth of activity. A ZIP with tickets in 2 weeks out of 90 is episodic, not chronic.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.first_ticket_ts IS
  'Earliest created_ts in this ZIP. Confirms the observation window start.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.last_ticket_ts IS
  'Latest created_ts in this ZIP. May be earlier than the citywide cutoff if the ZIP went quiet.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.how_to_read_temporal IS
  'Plain-language temporal caveats for this ZIP. Read before interpreting median close time or seasonal splits.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal.observation_cutoff IS
  'Inferred as-of timestamp: MAX(created_ts) on 311. Same as gold_zip_index.observation_cutoff.';
COMMENT ON COLUMN workspace.default.gold_zip_temporal._built_at IS
  'When this table was built.';

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_citywide_week AS
SELECT
  DATE_TRUNC('WEEK', created_ts) AS week_start,
  COUNT(*) AS n_created,
  SUM(CASE WHEN is_same_timestamp_close THEN 1 ELSE 0 END) AS n_same_timestamp_close,
  SUM(CASE WHEN status = 'In Progress' THEN 1 ELSE 0 END) AS n_still_in_progress,
  current_timestamp() AS _built_at
FROM workspace.default.silver_rat_sightings
WHERE created_ts IS NOT NULL
  AND complaint_type = 'Rodent'
GROUP BY DATE_TRUNC('WEEK', created_ts);

COMMENT ON TABLE workspace.default.gold_citywide_week IS
  'Citywide 311 created counts by week. Optional sparkline for viral-week caveat 10. Do NOT build a ZIP-week model. Do not treat a spike week as a cursed ZIP.';
COMMENT ON COLUMN workspace.default.gold_citywide_week.week_start IS
  'Week start of created_ts.';
COMMENT ON COLUMN workspace.default.gold_citywide_week.n_created IS
  'Citywide rodent 311 tickets created that week.';
COMMENT ON COLUMN workspace.default.gold_citywide_week.n_same_timestamp_close IS
  'Instant stamps created that week.';
COMMENT ON COLUMN workspace.default.gold_citywide_week.n_still_in_progress IS
  'Tickets created that week that are still In Progress at snapshot. Almost all leftover open is late-file weeks.';
COMMENT ON COLUMN workspace.default.gold_citywide_week._built_at IS
  'When this table was built.';

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_run_meta AS
SELECT
  (SELECT MAX(created_ts) FROM workspace.default.silver_rat_sightings) AS observation_cutoff,
  (SELECT MIN(created_ts) FROM workspace.default.silver_rat_sightings) AS created_min,
  DATE '2025-01-02' AS kitchen_window_start,
  DATE '2026-09-16' AS kitchen_window_end_inclusive,
  'Boundary-day completeness is unverified. Kitchen window is 2025-01-02 through 2026-09-16 inclusive (filter dt < 2026-09-17). 311 window is created_min through observation_cutoff.' AS window_note,
  COUNT(*) AS n_zips_observed,
  SUM(CASE WHEN is_rank_eligible THEN 1 ELSE 0 END) AS n_zips_rank_eligible,
  SUM(n_311) AS citywide_n_311_assigned_zip,
  SUM(n_mature) AS citywide_n_mature,
  SUM(n_timely_close_30d) AS citywide_n_timely_close_30d,
  SUM(n_gap_30d) AS citywide_n_gap_30d,
  SUM(n_gap_still_open) AS citywide_n_gap_still_open,
  SUM(n_gap_late_closed) AS citywide_n_gap_late_closed,
  SUM(n_k_instant) AS citywide_n_k_instant,
  100.0 * SUM(n_k_instant) / NULLIF(SUM(n_timely_close_30d), 0) AS pct_k_instant,
  ROUND(100.0 * SUM(n_k_instant) / NULLIF(SUM(n_timely_close_30d), 0), 1) AS pct_k_instant_display,
  100.0 * SUM(n_gap_30d) / NULLIF(SUM(n_mature), 0) AS citywide_sgi_30,
  ROUND(100.0 * SUM(n_gap_30d) / NULLIF(SUM(n_mature), 0), 1) AS citywide_sgi_30_display,
  AVG(CASE WHEN is_rank_eligible THEN sgi_30 END) AS citywide_sgi_unweighted_zip_mean,
  ROUND(AVG(CASE WHEN is_rank_eligible THEN sgi_30 END), 1) AS citywide_sgi_unweighted_zip_mean_display,
  SUM(n_same_timestamp_close) AS citywide_n_same_timestamp_close,
  SUM(n_backlog_14d) AS citywide_n_backlog_14d,
  SUM(n_restaurants) AS citywide_n_restaurants_on_spine,
  'SGI-30' AS metric_version,
  'SGI-30 = 100 * (N-K) / N for mature rodent 311 tickets. Mature = created at least 30 days before observation_cutoff = MAX(created_ts), not now(). K = recorded closure within 30 days of created and by the cutoff. Instant administrative stamps (closed_ts = created_ts) still count as K; they are not visits. HIDDEN TRAP: Closed% is ~95% and tens of thousands of tickets close at the same timestamp they opened — putting Closed% in SGI publishes the city always comes, which the records do not support. 10035 has the most 311 and tiny SGI because of one GPS cluster plus outreach, not best service. Kitchen 04K/04L shares barely track SGI; inspections are a parallel program, not the 311 response. Rank only zip_type neighborhood with N >= 30. Never AVG(sgi_30): unweighted ZIP mean overstates the pooled city gap. Packet missingness: no trust, tenure, language, population, HPD, visit logs, resolution text, or inspection_type. Quiet ZIP is not clean. Do not fetch ACS.' AS metric_definition,
  'Closed% and cursed-ZIP maps invert the truth; 10035 is a stress test; kitchens are not the 311 response.' AS hidden_trap,
  current_timestamp() AS _built_at
FROM workspace.default.gold_zip_index;

COMMENT ON TABLE workspace.default.gold_run_meta IS
  'ONE ROW. Snapshot rules for this gold build. Read this first. Observation cutoff is MAX(created_ts) on 311, not current_timestamp(). Citywide SGI is 100*SUM(N-K)/SUM(N) over all ZIPs with assigned ZIP, including ZIPs not on the leaderboard. citywide_sgi_unweighted_zip_mean is DO NOT USE — it overstates the gap vs pooled SGI. pct_k_instant is the share of on-time K that is a same-timestamp stamp.';
COMMENT ON COLUMN workspace.default.gold_run_meta.observation_cutoff IS
  'Inferred as-of timestamp: latest 311 created_ts in the extract. Use this to age tickets. Do not use now().';
COMMENT ON COLUMN workspace.default.gold_run_meta.created_min IS
  'Earliest 311 created_ts in the extract.';
COMMENT ON COLUMN workspace.default.gold_run_meta.kitchen_window_start IS
  'Shared cross-source window start (inspection dates). 2025-01-02.';
COMMENT ON COLUMN workspace.default.gold_run_meta.kitchen_window_end_inclusive IS
  'Shared window last included inspection date. 2026-09-16. Filter as dt < 2026-09-17.';
COMMENT ON COLUMN workspace.default.gold_run_meta.window_note IS
  'Disclose: boundary-day completeness is unverified.';
COMMENT ON COLUMN workspace.default.gold_run_meta.n_zips_observed IS
  'ZIPs seen in 311 or kitchens. Not every NYC ZIP.';
COMMENT ON COLUMN workspace.default.gold_run_meta.n_zips_rank_eligible IS
  'Neighborhood ZIPs with mature N >= 30.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_n_311_assigned_zip IS
  '311 tickets with a usable ZIP. Excludes quarantined 12345.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_n_mature IS
  'Citywide N. Includes ZIPs suppressed on the leaderboard.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_n_timely_close_30d IS
  'Citywide K.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_n_gap_30d IS
  'Citywide N-K.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_n_gap_still_open IS
  'Citywide mature tickets still open.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_n_gap_late_closed IS
  'Citywide mature tickets closed after 30 days.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_n_k_instant IS
  'Citywide on-time K that is an instant stamp.';
COMMENT ON COLUMN workspace.default.gold_run_meta.pct_k_instant IS
  'Share of citywide K that is a same-timestamp stamp. About half. Why Closed% is not SGI.';
COMMENT ON COLUMN workspace.default.gold_run_meta.pct_k_instant_display IS
  'pct_k_instant rounded to 1 decimal.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_sgi_30 IS
  'Pooled city SGI-30 = 100*SUM(N-K)/SUM(N). Do not average ZIP scores.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_sgi_30_display IS
  'Citywide SGI-30 to 1 decimal.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_sgi_unweighted_zip_mean IS
  'DO NOT USE. Mean of rank-eligible ZIP SGI values. Overstates the pooled gap. Genie must use citywide_sgi_30.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_sgi_unweighted_zip_mean_display IS
  'DO NOT USE. Display of the unweighted ZIP mean.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_n_same_timestamp_close IS
  'Tickets that closed at the same timestamp they were opened. Main reason Closed percent is not SGI.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_n_backlog_14d IS
  'In Progress and older than 14 days vs the cutoff.';
COMMENT ON COLUMN workspace.default.gold_run_meta.citywide_n_restaurants_on_spine IS
  'Sum of ZIP restaurant counts. Not a city distinct-camis total (do not sum across ZIPs for a city restaurant census).';
COMMENT ON COLUMN workspace.default.gold_run_meta.metric_version IS
  'SGI-30. If this name changes, do not silently reuse old numbers.';
COMMENT ON COLUMN workspace.default.gold_run_meta.metric_definition IS
  'Full definition, hidden trap, and packet missingness. Quote this when explaining the index.';
COMMENT ON COLUMN workspace.default.gold_run_meta.hidden_trap IS
  'One-liner: Closed% and cursed-ZIP maps invert the truth; 10035 is a stress test; kitchens are not the 311 response.';
COMMENT ON COLUMN workspace.default.gold_run_meta._built_at IS
  'When gold was built.';

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_lookup_contract AS
SELECT
  'gold_zip_index' AS zip_lookup_table,
  'gold_zip_leaderboard' AS top_n_table,
  'gold_borough_index' AS borough_table,
  'gold_run_meta' AS meta_table,
  'gold_zip_month' AS month_table,
  'gold_citywide_week' AS week_table,
  'Ranks are precomputed on gold_zip_index. Lookup a ZIP on the index. Top-N uses the leaderboard. Never RANK() after filtering to one ZIP. Absent ZIP = not in this extract, not invalid NYC ZIP.' AS rank_rule,
  'Citywide and borough SGI = 100 * SUM(N-K) / SUM(N). Never AVG(sgi_30). citywide_sgi_unweighted_zip_mean is labeled do not use.' AS sgi_aggregate_rule,
  'Use documented SGI-30 on gold. Do not re-derive Closed%. Instant stamps stay in K. High SGI = review administrative handling, not neglect, discrimination, infestation, or agency failure.' AS sgi_use_rule,
  'Attach gold only: gold_zip_index, gold_zip_leaderboard, gold_borough_index, gold_run_meta, gold_zip_month, gold_lookup_contract. Optional gold_citywide_week. Do not attach bronze or violation-grain kitchen rows.' AS attach_rule,
  'When demand is interpreted, name: trust, tenure, who stopped calling, HPD/landlord, language, and no ACS (caveats 1, 2, 4, 5, 8, 21). Quiet is not clean. Indoor mice are often HPD. zip_character is a card label; institution is not a campus; office_tourist is not hour-of-day; other is residual mix not location_category other; GPS cluster comes from cluster columns. 11430/11371/10069/zero-kitchen/zero-311: use sgi_status language. Ask only for a ZIP when the user says my block; otherwise use defaults and disclose them. Ground in queried gold; never invent population or neighboring ZIPs.' AS genie_instructions,
  'Hero is the ZIP lookup card, not a cursed-ZIP map. Typed ZIP from gold_zip_index: status, SGI or suppression, N/K/gap split, instant-close %, cluster/outreach/is_office_tourist_zip, zip_character, descriptor + location split (must include n_institution and n_sro), kitchens beside SGI, borough_sgi_30, how_to_read. Test 11430, 11371, 10069, 10005, 10035, 10128, 11423, 10463, 10001. Airports: kitchens only, few calls != good service. Thin / N<30: counts + borough number; no fake 0 or 100. 10035: stress test, one GPS cluster, not villain and not best-served; character can still be residential. 10001: office_tourist. Leaderboard if shown: largest 30-day paperwork gap among N>=30, N on every bar, small-N styling, never titled worst rats. Footer on every view: 311 measures who calls; Closed is not a visit; no ACS; ZIP is not a block.' AS dashboard_contract,
  current_timestamp() AS _built_at;

COMMENT ON TABLE workspace.default.gold_lookup_contract IS
  'ONE ROW. Genie and dashboard contract. Lookup = gold_zip_index (ranks precomputed, then filter). Top-N = gold_zip_leaderboard. Never rank after filtering to one ZIP. Absent ZIP = not in this extract, not invalid NYC ZIP. Attach gold only. Persist these instructions as Genie space instructions.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.zip_lookup_table IS
  'gold_zip_index. Always use this for a user ZIP, including airports and thin ZIPs.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.top_n_table IS
  'gold_zip_leaderboard. Rank-eligible only.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.borough_table IS
  'gold_borough_index. Pooled, not average of ZIP scores.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.meta_table IS
  'gold_run_meta. Read first.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.month_table IS
  'gold_zip_month. Do not average monthly SGI.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.week_table IS
  'gold_citywide_week. Citywide sparkline only.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.rank_rule IS
  'Ranks before filter. Missing ZIP is not invalid NYC.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.sgi_aggregate_rule IS
  'Pooled SUM, never AVG(sgi_30).';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.sgi_use_rule IS
  'Do not re-derive Closed%. High SGI is paperwork handling.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.attach_rule IS
  'Gold tables only. No bronze. No violation-grain kitchens.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.genie_instructions IS
  'Live-round Genie analyst instructions. Persist as space instructions.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract.dashboard_contract IS
  'ZIP-filter dashboard contract. Hero is the lookup card.';
COMMENT ON COLUMN workspace.default.gold_lookup_contract._built_at IS
  'When this row was built.';

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT zip, zip_type, zip_character, is_office_tourist_zip, n_311, n_mature, n_timely_close_30d,
       n_gap_30d, n_gap_still_open, n_gap_late_closed, n_k_instant, pct_same_timestamp_close,
       sgi_30_display, sgi_status, is_rank_eligible, is_small_n, borough_sgi_30_display,
       rank_sgi_citywide, n_restaurants, n_institution, n_sro, n_dwelling,
       is_cluster_dominant, max_cluster_share, is_stress_test_zip
FROM workspace.default.gold_zip_index
WHERE zip IN ('10035', '11430', '11371', '10069', '10001', '10314', '10452', '10005',
              '11233', '10020', '10128', '11423', '10463', '11370', '11103', '10116')
ORDER BY zip;

zip,zip_type,zip_character,is_office_tourist_zip,n_311,n_mature,n_timely_close_30d,n_gap_30d,n_gap_still_open,n_gap_late_closed,n_k_instant,pct_same_timestamp_close,sgi_30_display,sgi_status,is_rank_eligible,is_small_n,borough_sgi_30_display,rank_sgi_citywide,n_restaurants,n_institution,n_sro,n_dwelling,is_cluster_dominant,max_cluster_share,is_stress_test_zip
10001,neighborhood,office_tourist,true,55,50,49,1,0,1,35,71.4,2.0,published_ranked,true,true,2.3,116,619,1,0,11,false,0.073,false
10005,neighborhood,street_commercial,false,25,24,22,2,1,1,15,68.2,null,insufficient_data,false,true,2.3,null,52,0,0,6,false,0.2,false
10020,thin,office_tourist,true,1,1,1,0,0,0,1,100.0,null,insufficient_data,false,true,2.3,null,40,0,0,0,false,1.0,false
10035,neighborhood,residential,false,1501,1484,1480,4,0,4,166,11.2,0.3,published_ranked,true,false,2.3,148,83,3,0,1447,true,0.817,true
10069,neighborhood,street_commercial,false,14,12,12,0,0,0,6,50.0,null,insufficient_data,false,true,2.3,null,0,1,0,2,false,0.143,false
10116,building,building,false,0,0,0,0,0,0,0,null,null,no_311,false,true,null,null,3,0,0,0,false,null,false
10128,neighborhood,residential,false,531,510,508,2,0,2,263,51.8,0.4,published_ranked,true,false,2.3,144,146,3,0,391,false,0.081,false
10314,neighborhood,residential,false,285,260,229,31,13,18,154,67.2,11.9,published_ranked,true,false,12.2,22,187,2,0,223,false,0.032,false
10452,neighborhood,residential,false,1144,1118,1117,1,0,1,234,20.9,0.1,published_ranked,true,false,0.8,149,97,3,0,1006,true,0.323,false
10463,neighborhood,residential,false,465,441,423,18,5,13,222,52.5,4.1,published_ranked,true,false,null,91,122,3,0,349,false,0.082,false


In [0]:
%sql
SELECT
  SUM(n_311) AS n_311,
  SUM(n_dwelling + n_public_space + n_commercial + n_infrastructure + n_institution + n_vacant + n_other_location) AS location_sum,
  SUM(n_311) - SUM(n_dwelling + n_public_space + n_commercial + n_infrastructure + n_institution + n_vacant + n_other_location) AS location_gap,
  SUM(n_institution) AS n_institution,
  SUM(n_sro) AS n_sro
FROM workspace.default.gold_zip_index;

n_311,location_sum,location_gap,n_institution,n_sro
50953,50953,0,371,9


In [0]:
%sql
SELECT zip_character, COUNT(*) AS n_zips, SUM(n_311) AS tickets
FROM workspace.default.gold_zip_index
GROUP BY zip_character
ORDER BY n_zips DESC;

zip_character,n_zips,tickets
residential,138,45446
other,38,4134
building,30,8
street_commercial,8,853
office_tourist,5,509
airport,2,2
institution,1,1


In [0]:
%sql
SELECT observation_cutoff, citywide_n_mature, citywide_n_timely_close_30d, citywide_sgi_30_display,
       citywide_sgi_unweighted_zip_mean_display, pct_k_instant_display,
       citywide_n_gap_still_open, citywide_n_gap_late_closed,
       citywide_n_same_timestamp_close, n_zips_observed, n_zips_rank_eligible, hidden_trap
FROM workspace.default.gold_run_meta;

observation_cutoff,citywide_n_mature,citywide_n_timely_close_30d,citywide_sgi_30_display,citywide_sgi_unweighted_zip_mean_display,pct_k_instant_display,citywide_n_gap_still_open,citywide_n_gap_late_closed,citywide_n_same_timestamp_close,n_zips_observed,n_zips_rank_eligible,hidden_trap
2026-09-17T01:35:48.000Z,48046,45760,4.8,6.3,49.3,991,1295,22546,222,158,Closed% and cursed-ZIP maps invert the truth; 10035 is a stress test; kitchens are not the 311 response.


In [0]:
%sql
SELECT borough, n_mature, n_gap_still_open, n_gap_late_closed, n_k_instant,
       pct_same_timestamp_close, sgi_30_display, sgi_vs_city_display
FROM workspace.default.gold_borough_index
ORDER BY sgi_30 DESC;

borough,n_mature,n_gap_still_open,n_gap_late_closed,n_k_instant,pct_same_timestamp_close,sgi_30_display,sgi_vs_city_display
STATEN ISLAND,1856,153,74,999,61.3,12.2,7.5
QUEENS,8868,375,436,4651,57.7,9.1,4.4
BROOKLYN,17220,366,539,8217,50.4,5.3,0.5
MANHATTAN,12476,82,200,4697,38.5,2.3,-2.5
BRONX,7627,15,46,3983,52.6,0.8,-4.0


In [0]:
%sql
SELECT sgi_status, COUNT(*) AS n_zips, SUM(n_311) AS tickets
FROM workspace.default.gold_zip_index
GROUP BY sgi_status
ORDER BY n_zips DESC;

sgi_status,n_zips,tickets
published_ranked,158,50580
no_311,33,0
insufficient_data,31,373


In [0]:
%sql
SELECT zip, borough, sgi_30_display, rank_sgi_citywide, rank_sgi_borough, n_mature, n_311, is_small_n, zip_character
FROM workspace.default.gold_zip_leaderboard
ORDER BY rank_sgi_citywide
LIMIT 10;

zip,borough,sgi_30_display,rank_sgi_citywide,rank_sgi_borough,n_mature,n_311,is_small_n,zip_character
11423,QUEENS,29.5,1,1,44,47,true,residential
11415,QUEENS,22.0,2,2,59,63,true,residential
11694,QUEENS,21.9,3,3,64,66,true,other
10308,STATEN ISLAND,21.3,4,1,136,143,false,residential
11416,QUEENS,20.7,5,4,58,62,true,residential
10306,STATEN ISLAND,18.9,6,2,244,268,false,residential
11361,QUEENS,18.4,7,5,76,82,true,residential
11413,QUEENS,18.0,8,6,128,136,false,residential
11411,QUEENS,16.7,9,7,48,49,true,residential
11436,QUEENS,16.1,10,8,87,92,true,residential


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_glossary AS
SELECT term, definition, related_columns
FROM VALUES
  ('SGI-30',
   'The share of rodent complaints filed more than 30 days ago that still have no recorded closure. Higher means more people called and nothing happened on paper.',
   'sgi_30, sgi_30_display, rank_sgi_citywide'),

  ('Mature ticket',
   'A complaint filed more than 30 days ago. Only complaints old enough for the city to have acted on are scored — anything newer is too early to judge.',
   'n_mature, n_younger_than_30d'),

  ('N',
   'The number of complaints in a ZIP that are old enough to be scored. A ZIP needs at least 30 to get a published score.',
   'n_mature'),

  ('K',
   'The number of scoreable complaints that got a recorded closure within 30 days.',
   'n_timely_close_30d'),

  ('Right-censoring',
   'Complaints filed in the last 30 days are left out of the score so the city is not penalized for tickets that are still brand new.',
   'n_younger_than_30d'),

  ('Same-timestamp close',
   'A complaint marked closed at the exact same second it was opened. Could be a duplicate, a bounce, or an auto-close — not necessarily someone showing up.',
   'n_same_timestamp_close'),

  ('CAMIS',
   'A unique ID the Health Department gives each restaurant. Restaurants are counted by this ID, not by name or by rows in the file.',
   'n_restaurants'),

  ('04K',
   'A health inspection code meaning the inspector found evidence of rats in a restaurant kitchen.',
   'n_restaurants_04k, pct_establishments_04k'),

  ('04L',
   'A health inspection code meaning the inspector found evidence of mice. Different animal from 04K — they should not be mixed.',
   'n_restaurants_04l, pct_establishments_04l'),

  ('08A',
   'A health inspection code for conditions that attract pests — holes, gaps, clutter. Not a live rodent sighting.',
   'n_restaurants_08a, pct_establishments_08a'),

  ('Cluster dominant',
   'A ZIP where more than 30% of all complaints come from one GPS coordinate. Usually means one building or one site, not an entire neighborhood in trouble.',
   'is_cluster_dominant, max_cluster_share'),

  ('Rank eligible',
   'A ZIP that qualifies for the leaderboard. It must be a real neighborhood with at least 30 scoreable complaints. Airports and thin-data ZIPs are left out.',
   'is_rank_eligible, sgi_status'),

  ('Observation cutoff',
   'The timestamp of the most recent complaint in the data. Everything is measured relative to this point, not the current clock.',
   'observation_cutoff'),

  ('Pooled SGI',
   'Borough and citywide scores are computed by adding up all complaints and closures first, then dividing. ZIP scores are not averaged — that would give a tiny ZIP the same weight as a large one.',
   'gold_borough_index.sgi_30, gold_run_meta.citywide_sgi_30'),

  ('Establishment-date proxy',
   'Each restaurant ID paired with each inspection date, used to estimate how many inspection visits happened. It is an approximation because the data has no unique visit ID.',
   'n_visits')

AS t(term, definition, related_columns);

COMMENT ON TABLE workspace.default.gold_glossary IS
  'Glossary of key terms in plain language. Genie should explain any term from this table inline when it appears in a response.';
COMMENT ON COLUMN workspace.default.gold_glossary.term IS 'The term being defined.';
COMMENT ON COLUMN workspace.default.gold_glossary.definition IS 'Plain-language explanation using no internal jargon.';
COMMENT ON COLUMN workspace.default.gold_glossary.related_columns IS 'Gold layer columns where this concept appears.';

num_affected_rows,num_inserted_rows


## Gold contract

- **Question:** when someone calls, does the city show up? **Answer we can support:** we can see paperwork clocks, who still files 311, GPS pile-ups, and kitchen citations in the same ZIP. We cannot see a visit.
- **SGI-30** lives here, not in silver. Do not put Closed% of all tickets in the index. Instant stamps stay in K.
- **Kitchens sit beside SGI**, not inside it. No causal arrow.
- **Leaderboard** = `gold_zip_leaderboard` only. ZIP lookup = `gold_zip_index` (so a filtered ZIP does not rank #1).
- **10035** = stress test. One GPS cluster. Outreach ZIP. Tiny SGI does not mean best served. Character can still be `residential`.
- **10128** = neighborhood (UES), not a 101xx building.
- **zip_character** is a card label. `zip_type` is the rank gate. Do not call `institution` a campus.
- Rebuild comments after every `CREATE OR REPLACE`.

## Dashboard (ZIP Code Challenge)

Hero is the **typed ZIP card**, not a cursed-ZIP map. Filter `gold_zip_index.zip`. Footer on every view: 311 measures who calls; Closed is not a visit; no ACS; ZIP is not a block.

Test ZIPs: 11430, 11371, 10069, 10005, 10035, 10128, 11423, 10463, 10001.

## Genie

Attach gold only. Persist `gold_lookup_contract.genie_instructions` as space instructions. Read `gold_run_meta` first.

## Finding (story)

If I worked for the city, I would **not send extra crews to the ZIP with the most 311 calls or the highest SGI-30 rank**, because our data shows **311 is who still files, SGI-30 is a 30-day paperwork clock (about half of on-time closes are same-timestamp stamps), and 10035 looks “best served” only as an outreach-plus-one-cluster stress test — we cannot see whether anyone showed up.**

Biggest limitation: resolution text, visit logs, HPD, population, and language are not in the packet.